In [ ]:
#!pip install firebase_admin

In [3]:
import sqlite3
import firebase_admin
from firebase_admin import credentials, firestore
import json

In [4]:
# --- Configuration ---
# Replace with the path to your SQLite database file
SQLITE_DB_PATH = './Streamlit Interface/real_estate.db'

# Replace with the path to your Firebase service account key JSON file
SERVICE_ACCOUNT_KEY_PATH = 'real-estate-parser-f44a0-firebase-adminsdk-fbsvc-56cddcced1.json'

# Initialize Firebase Admin SDK
try:
    cred = credentials.Certificate(SERVICE_ACCOUNT_KEY_PATH)
    firebase_admin.initialize_app(cred)
    db = firestore.client()
    print("Firebase Admin SDK initialized successfully.")
except Exception as e:
    print(f"Error initializing Firebase Admin SDK: {e}")
    exit()

Firebase Admin SDK initialized successfully.


In [ ]:

def upload_sqlite_to_firestore(sqlite_db_path, collection_name):
    """
    Reads data from an SQLite table and uploads it to a Firestore collection.

    Args:
        sqlite_db_path (str): Path to the SQLite database file.
        collection_name (str): The name of the Firestore collection to create/update.
    """
    try:
        conn = sqlite3.connect(sqlite_db_path)
        cursor = conn.cursor()
        print(f"Connected to SQLite database: {sqlite_db_path}")

        cursor.execute(f"SELECT * FROM {collection_name}")
        rows = cursor.fetchall()
        column_names = [description[0] for description in cursor.description]

        if not rows:
            print(f"No data found in SQLite table '{collection_name}'.")
            return

        print(f"Found {len(rows)} rows in SQLite table '{collection_name}'. Uploading to Firestore...")

        batch = db.batch()
        doc_count = 0

        for row in rows:
            document_data = {}
            # Map SQLite columns to Firestore document fields
            for i, column_name in enumerate(column_names):
                document_data[column_name] = row[i]

            # Use a unique identifier for each document, e.g., an 'id' column if available
            # If not, Firestore will generate an automatic ID.
            # For this example, we'll let Firestore generate an ID.
            doc_ref = db.collection(collection_name).document()
            batch.set(doc_ref, document_data)
            doc_count += 1

            # Commit batch every 500 documents (Firestore batch write limit)
            if doc_count % 500 == 0:
                batch.commit()
                batch = db.batch() # Start a new batch
                print(f"Uploaded {doc_count} documents so far...")

        # Commit any remaining documents in the last batch
        batch.commit()
        print(f"Successfully uploaded {doc_count} documents to Firestore collection '{collection_name}'.")

    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        if 'conn' in locals() and conn:
            conn.close()
            print("SQLite connection closed.")

In [7]:
upload_sqlite_to_firestore(SQLITE_DB_PATH, 'listings')

Connected to SQLite database: ./Streamlit Interface/real_estate.db
Found 1091 rows in SQLite table 'listings'. Uploading to Firestore...
Uploaded 500 documents so far...
Uploaded 1000 documents so far...
Successfully uploaded 1091 documents to Firestore collection 'listings'.
SQLite connection closed.
